# Control del Espacio en Futbol: Voronoi y Pitch Control

Un partido de futbol es una pelea por el espacio. El balon mide 22 cm, pero el campo tiene 7.140 m² donde 22 jugadores compiten por dominar cada zona.

En este articulo implementamos dos modelos de control espacial desde cero:

1. **Diagramas de Voronoi** — el modelo mas simple: cada punto del campo pertenece al jugador mas cercano
2. **Pitch Control (PPCF)** — Spearman 2018: quien tiene mas probabilidad de controlar cada punto, teniendo en cuenta velocidad, tiempo de reaccion e incertidumbre

Usamos **tracking oficial de Bundesliga a 25Hz** (Bassek et al. 2025, Nature Scientific Data) — posiciones exactas de todos los jugadores y el balon, con velocidades y aceleraciones, 25 veces por segundo.

---

**Datos**: DFL Bundesliga Open Dataset — 1. FC Koln vs FC Bayern Munchen, Jornada 34, Temporada 2022/23

**Referencia**: Spearman (2018) *"Beyond Expected Goals"*

In [ ]:
import sys, warnings
from pathlib import Path

# Meter la raiz del proyecto en el path para que resuelva "blog.pitch_control"
PROJECT_ROOT = str(Path.cwd().parents[1]) if Path.cwd().name == "pitch_control" else str(Path.cwd())
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import HTML

# Nuestros modulos
from blog.pitch_control.loader import (
    load_match_info, load_tracking, load_tracking_frame, load_events,
    get_frame_at_time, get_player_lookup, get_team_lookup, get_goalkeeper_ids,
)
from blog.pitch_control.pitch_control import (
    compute_velocities, compute_ppcf, compute_ppcf_sequence, default_model_params,
)
from blog.pitch_control.viz import (
    make_pitch, draw_pitch, plot_players, plot_voronoi, plot_ppcf,
    plot_frame_voronoi, plot_frame_ppcf, plot_comparison,
    BG, WHITE, ATT, DEF, PPCF_CMAP,
)
from blog.pitch_control.video import render_ppcf_video

## 1. Cargar los datos

Usamos el **Koln vs Bayern** (jornada 34, 2022/23) del dataset abierto de la DFL. El tracking nos da las posiciones de los 22 jugadores + balon a 25 fotogramas por segundo, con velocidad (m/s) y direccion (grados) ya calculados por el sistema TRACAB.

Las coordenadas estan en metros, centradas en (0, 0) — es decir, el centro del campo es el origen.

In [ ]:
MATCH_ID = "DFL-MAT-J03WMX"

# Cargar info del partido
mi = load_match_info(MATCH_ID)
names = get_player_lookup(mi)   # id -> nombre corto
teams = get_team_lookup(mi)     # id -> nombre equipo

HOME = mi.home_team.team_id    # Koln
AWAY = mi.guest_team.team_id   # Bayern

print(f"{mi.home_team.team_name} vs {mi.guest_team.team_name}")
print(f"Resultado: {mi.result} | {mi.stadium} | Temporada {mi.season}, J{mi.match_day}")
print(f"Campo: {mi.pitch_x}m x {mi.pitch_y}m")
print(f"Formacion local: {mi.home_team.lineup} | Visitante: {mi.guest_team.lineup}")

In [ ]:
# Cargar tracking de la primera parte a 25Hz completo
tracking = load_tracking(MATCH_ID, game_section="firstHalf", subsample=1)
# Derivar vx, vy a partir de velocidad + direccion del tracking
tracking = compute_velocities(tracking)
# Detectar porteros por posicion media (el mas pegado a su porteria)
gk_ids = get_goalkeeper_ids(mi, tracking)

gk_names = [names.get(pid, "?") for pid in gk_ids]
print(f"Tracking: {tracking.shape[0]:,} filas | {tracking['frame'].nunique():,} frames | 25 Hz")
print(f"Jugadores por frame: {tracking[~tracking['is_ball']].groupby('frame').size().median():.0f}")
print(f"Rango temporal: {tracking['time_s'].min():.0f}s - {tracking['time_s'].max():.0f}s")
print(f"Porteros detectados: {gk_names}")
print(f"Memoria: {tracking.memory_usage(deep=True).sum()/1e6:.0f} MB")

### Un frame suelto

Vamos a pillar un momento del minuto 10 — una jugada normal de juego abierto. Cada frame tiene 22 jugadores (11 por equipo) mas el balon, con posiciones x/y y flechas de velocidad.

In [ ]:
# Buscar el frame mas cercano al minuto 10
frame_num = get_frame_at_time(tracking, time_s=600)
frame = load_tracking_frame(tracking, frame_num)

print(f"Frame {frame_num} | t = {frame['time_s'].iloc[0]:.1f}s ({frame['time_s'].iloc[0]/60:.1f} min)")
print(f"Jugadores: {(~frame['is_ball']).sum()} | Balon: {frame['is_ball'].sum()}")

# Dibujar campo con jugadores y flechas de velocidad
fig, ax = make_pitch()
plot_players(ax, frame, att_team_id=HOME, gk_ids=gk_ids, names=names, show_names=True)
ax.set_title(
    f"{mi.home_team.team_name} (azul) vs {mi.guest_team.team_name} (rojo) — min {frame['time_s'].iloc[0]/60:.0f}'",
    color=WHITE, fontsize=14, fontweight="bold", pad=10,
)
plt.show()

## 2. Diagramas de Voronoi — El modelo mas basico

Un **diagrama de Voronoi** parte el campo en zonas: cada punto pertenece al jugador mas cercano. Asi de simple.

$$\text{Voronoi}(p_i) = \{ x \in \text{Campo} \mid \|x - p_i\| \leq \|x - p_j\| \; \forall j \neq i \}$$

El problema es obvio: trata a todos los jugadores igual. Da lo mismo que uno este esprintando hacia una zona y otro caminando en sentido contrario — si estan a la misma distancia, les toca el mismo trozo. Las fronteras son binarias, no hay zonas disputadas.

In [ ]:
# Voronoi sobre el mismo frame
fig, ax = plot_frame_voronoi(
    frame, att_team_id=HOME, gk_ids=gk_ids, names=names,
    title=f"Voronoi — {mi.home_team.team_name} vs {mi.guest_team.team_name}",
)
plt.show()

El Voronoi nos dice *quien esta mas cerca*, pero no *quien tiene mas probabilidad de llegar primero*. Un defensa esprintando a 8 m/s hacia una zona deberia controlar mas espacio del que le asigna el Voronoi. Necesitamos un modelo que tenga en cuenta la **velocidad** y la **incertidumbre**.

---

## 3. Pitch Control (PPCF) — Spearman 2018

El **Pitch Control** (Spearman 2018) calcula la probabilidad de que cada equipo controle cada punto del campo. Para cada posicion objetivo $x$:

**Tiempo de llegada** del jugador $i$:

$$T_i(x) = t_{\text{reac}} + \frac{\|x - (p_i + v_i \cdot t_{\text{reac}})\|}{v_{\max}}$$

Durante el tiempo de reaccion el jugador sigue con su velocidad actual. Despues, esprintando a tope hacia el objetivo.

**Probabilidad de haber llegado** en el instante $\tau$ (sigmoide):

$$\rho_i(\tau, x) = \frac{1}{1 + e^{-\frac{\pi}{\sqrt{3}\sigma}(\tau - T_i(x))}}$$

**Control acumulado** por integracion de Euler:

$$\text{PPCF}_{\text{att}}(x) = \sum_i \int_0^{\infty} \lambda_i \cdot \rho_i(\tau, x) \cdot (1 - \text{PPCF}_{\text{att}} - \text{PPCF}_{\text{def}}) \, d\tau$$

Parametros clave:
- $v_{\max} = 5$ m/s — velocidad maxima en el modelo
- $t_{\text{reac}} = 0.7$ s — tiempo de reaccion
- $\sigma = 0.45$ s — incertidumbre en el tiempo de llegada
- $\lambda_{\text{att}} = 4.3$ — tasa de control del balon

La gracia: un jugador que corre **hacia** una zona llega antes y controla mas espacio. Uno que corre en sentido contrario, pierde territorio. Eso es lo que el Voronoi no puede capturar.

In [ ]:
# Calcular PPCF con las velocidades reales del tracking
ppcf, xgrid, ygrid = compute_ppcf(
    frame, att_team_id=HOME, gk_ids=gk_ids, n_grid_x=60,
)

# Pintar superficie de control
fig, ax = plot_frame_ppcf(
    frame, ppcf, xgrid, ygrid,
    att_team_id=HOME, gk_ids=gk_ids, names=names,
    title="Pitch Control (PPCF) — Spearman 2018",
)
plt.show()

# Porcentaje de campo controlado
att_pct = (ppcf > 0.5).mean() * 100
print(f"Koln controla {att_pct:.0f}% del campo (PPCF > 0.5)")

---

## 4. Video — El control del espacio en movimiento

Lo mejor del pitch control es verlo evolucionar en tiempo real. Renderizamos dos clips de 30 segundos: uno con el Voronoi y otro con el PPCF, cada uno por separado para que se vean bien.

In [ ]:
# Ventana de 30 segundos alrededor del minuto 10
t_start = 600
t_end = 630
f_start = get_frame_at_time(tracking, t_start)
f_end = get_frame_at_time(tracking, t_end)
print(f"Ventana: frames {f_start} - {f_end} ({f_end - f_start} frames a 25Hz)")

# Pre-calcular PPCF para cada frame de la ventana (para el video de PPCF)
ppcf_dict = compute_ppcf_sequence(
    tracking, att_team_id=HOME,
    frame_range=(f_start, f_end), every_n=1,
    gk_ids=gk_ids, n_grid_x=50,
)
print(f"Superficies PPCF calculadas: {len(ppcf_dict)}")

In [ ]:
# Video 1: Voronoi animado
voronoi_path = "blog/pitch_control/voronoi_clip.mp4"
anim_vor = render_ppcf_video(
    tracking, att_team_id=HOME, ppcf_dict={},
    frame_range=(f_start, f_end),
    gk_ids=gk_ids, names=names,
    title=f"Voronoi — {mi.home_team.team_name} vs {mi.guest_team.team_name}",
    mode="voronoi", fps=25, save_path=voronoi_path,
)
plt.close()
print(f"Video Voronoi guardado en {voronoi_path}")

In [ ]:
# Mostrar video Voronoi
from IPython.display import Video
Video(voronoi_path, embed=True, width=900)

In [ ]:
# Video 2: PPCF animado
ppcf_path = "blog/pitch_control/ppcf_clip.mp4"
anim_ppcf = render_ppcf_video(
    tracking, att_team_id=HOME, ppcf_dict=ppcf_dict,
    frame_range=(f_start, f_end),
    gk_ids=gk_ids, names=names,
    title=f"Pitch Control — {mi.home_team.team_name} vs {mi.guest_team.team_name}",
    mode="ppcf", fps=25, save_path=ppcf_path,
)
plt.close()
print(f"Video PPCF guardado en {ppcf_path}")

In [ ]:
# Mostrar video PPCF
Video(ppcf_path, embed=True, width=900)

---

## Resumen

| Modelo | Que captura | Que se le escapa |
|--------|------------|------------------|
| **Voronoi** | Jugador mas cercano a cada punto | Velocidad, direccion, incertidumbre |
| **PPCF** | Probabilidad real de control (velocidad + reaccion + incertidumbre) | Fatiga, intencion tactica, trayectoria del balon |

**Lo importante:**

1. El Voronoi es util para un vistazo rapido, pero es ciego al movimiento. Un jugador parado y uno esprintando reciben el mismo territorio si estan a la misma distancia.

2. El PPCF da **fronteras blandas** donde las zonas disputadas se ven en gris, y desplaza el control en la direccion en la que corre cada jugador.

3. Con tracking a 25Hz podemos calcular el pitch control en cada frame y ver como evoluciona el dominio espacial en tiempo real — algo que la posesion clasica no puede contar.

---

**Codigo**: `blog/pitch_control/` — loader, pitch_control, viz, video.

**Datos**: DFL Bundesliga Open Dataset (Bassek et al. 2025, Nature Scientific Data).

**Referencia**: Spearman (2018) *"Beyond Expected Goals"*.